In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

In [19]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # En Jupyter local no es necesario montar Drive.

posibles_rutas = [
    Path('/content/drive/MyDrive/mineriaDatos/online_shoppers_intention.csv'),
    Path('online_shoppers_intention.csv'),
    Path('/mnt/data/online_shoppers_intention.csv'),
]
archivo = next((p for p in posibles_rutas if p.exists()), None)
if archivo is None:
    raise FileNotFoundError('Coloca online_shoppers_intention.csv en la carpeta mineriaDatos de Drive o junto al notebook.')

df = pd.read_csv(archivo)
df_original = df.copy()
print('Archivo:', archivo)
print('Dimensiones originales:', df_original.shape)
display(df_original.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivo: /content/drive/MyDrive/mineriaDatos/online_shoppers_intention.csv
Dimensiones originales: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


###  Comprobar faltantes y duplicados

**este dataset no tiene valores faltantes**. Por ello no aplicaremos `fillna()` sin necesidad. Es importante no alterar datos correctos.

In [20]:
faltantes = pd.DataFrame({
    'Cantidad': df_original.isna().sum(),
    'Porcentaje': (df_original.isna().mean() * 100).round(2)
})
print('Valores faltantes totales:', int(df_original.isna().sum().sum()))
display(faltantes)

print('Filas duplicadas adicionales:', int(df_original.duplicated().sum()))
print('Filas involucradas en grupos duplicados:', int(df_original.duplicated(keep=False).sum()))
print('Revenue de los duplicados adicionales:')
print(df_original.loc[df_original.duplicated(), 'Revenue'].value_counts())

Valores faltantes totales: 0


,Cantidad,Porcentaje
Administrative,0,0.0
Administrative_Duration,0,0.0
Informational,0,0.0
Informational_Duration,0,0.0
ProductRelated,0,0.0
ProductRelated_Duration,0,0.0
BounceRates,0,0.0
ExitRates,0,0.0
PageValues,0,0.0
SpecialDay,0,0.0


Filas duplicadas adicionales: 125
Filas involucradas en grupos duplicados: 201
Revenue de los duplicados adicionales:
Revenue
False    125
Name: count, dtype: int64


### Crear una versión sin duplicados

En el dataset no existe un identificador único de sesión. Dos sesiones distintas podrían tener los mismos valores. Por eso conservamos dos versiones:

- `df_original`: todas las sesiones del CSV (sin modificar).
- `df_sin_duplicados`: se conserva la primera aparición de cada fila y se elimina el resto.

**Decisión inicial para los modelos:** usar la versión sin duplicados, para evitar que copias idénticas de una fila puedan quedar en entrenamiento y prueba. Más adelante comprobaremos si conservarlas cambia los resultados; no podemos asegurar que todas sean errores de captura.

In [21]:
df_sin_duplicados = df_original.drop_duplicates(keep='first').reset_index(drop=True)

comparacion = pd.DataFrame({
    'Dataset': ['Original', 'Sin duplicados'],
    'Sesiones': [len(df_original), len(df_sin_duplicados)],
    'Compras': [int(df_original['Revenue'].sum()), int(df_sin_duplicados['Revenue'].sum())],
    'Porcentaje de compra': [df_original['Revenue'].mean() * 100,
                             df_sin_duplicados['Revenue'].mean() * 100]
})
display(comparacion.round(2))

# Cambiar a False solo cuando hagamos la comparación de sensibilidad.
ELIMINAR_DUPLICADOS = True

df_trabajo = (df_sin_duplicados.copy() if ELIMINAR_DUPLICADOS
              else df_original.copy())
print('Dataset elegido para los siguientes pasos:', df_trabajo.shape)

,Dataset,Sesiones,Compras,Porcentaje de compra
0,Original,12330,1908,15.47
1,Sin duplicados,12205,1908,15.63


Dataset elegido para los siguientes pasos: (12205, 18)


###  Verificar tipos de variables y categorías

Aunque `OperatingSystems`, `Browser`, `Region` y `TrafficType` están guardadas como enteros, son **códigos categóricos**. `SpecialDay` es una escala numérica de proximidad a una fecha especial; no la convertimos en etiqueta.

In [22]:
variables_numericas = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]

variables_categoricas = [
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend', 'Revenue'
]

for variable in variables_categoricas:
    print(f'{variable}: {df_trabajo[variable].nunique()} categorías')
    print(df_trabajo[variable].value_counts().head(10))
    print('-' * 35)

Month: 10 categorías
Month
May     3329
Nov     2982
Mar     1860
Dec     1706
Oct      549
Sep      448
Aug      433
Jul      432
June     285
Feb      181
Name: count, dtype: int64
-----------------------------------
OperatingSystems: 8 categorías
OperatingSystems
2    6541
1    2549
3    2530
4     478
8      75
6      19
7       7
5       6
Name: count, dtype: int64
-----------------------------------
Browser: 13 categorías
Browser
2     7883
1     2427
4      731
5      465
6      174
10     163
8      135
3      105
13      56
7       49
Name: count, dtype: int64
-----------------------------------
Region: 9 categorías
Region
1    4714
3    2379
4    1171
2    1128
6     801
7     758
9     505
8     431
5     318
Name: count, dtype: int64
-----------------------------------
TrafficType: 20 categorías
TrafficType
2     3911
1     2388
3     2013
4     1066
13     728
10     450
6      443
8      343
5      260
11     247
Name: count, dtype: int64
---------------------------------

### Revisar valores fuera de rangos posibles

Para detectar errores básicos, revisamos si hay conteos, duraciones y valores negativos o tasas fuera de sus rangos admitidos.

In [23]:
columnas_no_negativas = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration', 'PageValues'
]

for variable in columnas_no_negativas:
    print(variable, '- valores negativos:', int((df_trabajo[variable] < 0).sum()))

for variable in ['BounceRates', 'ExitRates', 'SpecialDay']:
    fuera_de_rango = ~df_trabajo[variable].between(0, 1)
    print(variable, '- fuera de [0,1]:', int(fuera_de_rango.sum()))

Administrative - valores negativos: 0
Administrative_Duration - valores negativos: 0
Informational - valores negativos: 0
Informational_Duration - valores negativos: 0
ProductRelated - valores negativos: 0
ProductRelated_Duration - valores negativos: 0
PageValues - valores negativos: 0
BounceRates - fuera de [0,1]: 0
ExitRates - fuera de [0,1]: 0
SpecialDay - fuera de [0,1]: 0


###  Detectar posibles valores atipicos mediante el IQR


Cuando el IQR vale cero, un valor positivo podría aparecer marcado como atípico aunque sea completamente válido. No eliminamos ni recortamos automáticamente estas observaciones.

In [24]:
def detectar_outliers_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    mascara = (serie < limite_inferior) | (serie > limite_superior)
    return mascara, limite_inferior, limite_superior, iqr

filas_outliers = []
for variable in variables_numericas:
    mascara, li, ls, iqr = detectar_outliers_iqr(df_trabajo[variable])
    filas_outliers.append({
        'Variable': variable,
        'IQR': iqr,
        'Límite inferior': li,
        'Límite superior': ls,
        'Posibles outliers': int(mascara.sum()),
        'Porcentaje': 100 * mascara.mean()
    })

tabla_outliers = pd.DataFrame(filas_outliers)
display(tabla_outliers.round(2))

,Variable,IQR,Límite inferior,Límite superior,Posibles outliers,Porcentaje
0,Administrative,4.00,-6.00,10.00,404,3.31
1,Administrative_Duration,94.70,-142.05,236.75,1149,9.41
2,Informational,0.00,0.00,0.00,2631,21.56
3,Informational_Duration,0.00,0.00,0.00,2405,19.71
4,ProductRelated,30.00,-37.00,83.00,1007,8.25
5,ProductRelated_Duration,1284.15,-1733.23,3403.39,951,7.79
6,BounceRates,0.02,-0.03,0.04,1428,11.70
7,ExitRates,0.03,-0.04,0.10,1325,10.86
8,PageValues,0.00,0.00,0.00,2730,22.37
9,SpecialDay,0.00,0.00,0.00,1249,10.23


###  Revisar la consistencia del texto

Comprobamos las etiquetas de mes y tipo de visitante.

In [25]:
for variable in ['Month', 'VisitorType']:
    print(f'Valores únicos de {variable}:')
    print(sorted(df_trabajo[variable].unique()))
    print()

# Conservar los valores originales: no se encontraron nulos ni nombres
# alternativos demostrados para estas dos columnas.

Valores únicos de Month:
['Aug', 'Dec', 'Feb', 'Jul', 'June', 'Mar', 'May', 'Nov', 'Oct', 'Sep']

Valores únicos de VisitorType:
['New_Visitor', 'Other', 'Returning_Visitor']



###  Variables nuevas para describir navegación
Para **este** dataset de comercio electrónico, una alternativa coherente es contar el total de páginas visitadas y si hubo consultas informativas. No se añadirán automáticamente como predictores hasta decidir si son útiles.

In [26]:
df_trabajo['PaginasTotales'] = (df_trabajo['Administrative']
                             + df_trabajo['Informational']
                             + df_trabajo['ProductRelated'])
df_trabajo['VisitoInformativas'] = (df_trabajo['Informational'] > 0).astype(int)

display(df_trabajo[['Administrative', 'Informational', 'ProductRelated',
                    'PaginasTotales', 'VisitoInformativas']].head())

,Administrative,Informational,ProductRelated,PaginasTotales,VisitoInformativas
0,0,0,1,1,0
1,0,0,2,2,0
2,0,0,1,1,0
3,0,0,2,2,0
4,0,0,10,10,0


##  Preparación para **clasificación** (Revenue)

**Variable objetivo:** `Revenue` (0 = no compró; 1 = compró). Usaremos regresión logística

|

In [27]:
columnas_para_clasificacion = variables_numericas + [
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend'
]
columnas_para_clasificacion.remove('PageValues')

X = df_trabajo[columnas_para_clasificacion].copy()
Y = df_trabajo['Revenue'].astype(int)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, stratify=Y
)

print('Entrenamiento:', X_train.shape, 'Prueba:', X_test.shape)
print('Proporción de compras, entrenamiento:', round(Y_train.mean() * 100, 2), '%')
print('Proporción de compras, prueba:', round(Y_test.mean() * 100, 2), '%')

Entrenamiento: (9764, 16) Prueba: (2441, 16)
Proporción de compras, entrenamiento: 15.63 %
Proporción de compras, prueba: 15.65 %


###  One-Hot Encoding como en los ejercicios de clase

Con `pd.get_dummies()` representamos las categorías sin imponer un orden ficticio. Las columnas de prueba se alinean con las de entrenamiento. Si en prueba aparece una categoría no vista al entrenar, sus indicadores quedarán en cero y esta limitación se documentará.

In [28]:
columnas_onehot = [
    'Month', 'OperatingSystems', 'Browser',
    'Region', 'TrafficType', 'VisitorType'
]

X_train_aux = X_train.copy()
X_test_aux = X_test.copy()

for variable in columnas_onehot:
    X_train_aux[variable] = X_train_aux[variable].astype(str)
    X_test_aux[variable] = X_test_aux[variable].astype(str)

X_train_aux['Weekend'] = X_train_aux['Weekend'].astype(int)
X_test_aux['Weekend'] = X_test_aux['Weekend'].astype(int)

X_train_cod = pd.get_dummies(X_train_aux, columns=columnas_onehot, dtype=int)
X_test_cod = pd.get_dummies(X_test_aux, columns=columnas_onehot, dtype=int)
X_test_cod = X_test_cod.reindex(columns=X_train_cod.columns, fill_value=0)

print('Dimensiones después de codificar:')
print('Entrenamiento:', X_train_cod.shape, 'Prueba:', X_test_cod.shape)
print('Mismas columnas:', X_train_cod.columns.equals(X_test_cod.columns))
display(X_train_cod.head())

Dimensiones después de codificar:
Entrenamiento: (9764, 72) Prueba: (2441, 72)
Mismas columnas: True


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,SpecialDay,Weekend,Month_Aug,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,OperatingSystems_1,OperatingSystems_2,OperatingSystems_3,OperatingSystems_4,OperatingSystems_5,OperatingSystems_6,OperatingSystems_7,OperatingSystems_8,Browser_1,Browser_10,Browser_11,Browser_12,Browser_13,Browser_2,Browser_3,Browser_4,Browser_5,Browser_6,Browser_7,Browser_8,Browser_9,Region_1,Region_2,Region_3,Region_4,Region_5,Region_6,Region_7,Region_8,Region_9,TrafficType_1,TrafficType_10,TrafficType_11,TrafficType_13,TrafficType_14,TrafficType_15,TrafficType_16,TrafficType_17,TrafficType_18,TrafficType_19,TrafficType_2,TrafficType_20,TrafficType_3,TrafficType_4,TrafficType_5,TrafficType_6,TrafficType_7,TrafficType_8,TrafficType_9,VisitorType_New_Visitor,VisitorType_Other,VisitorType_Returning_Visitor
7469,8,203.6,1,0.0,42,1052.400000,0.0,0.004545,0.0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1
11804,0,0.0,0,0.0,30,901.288889,0.0,0.007333,0.0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0
3250,0,0.0,0,0.0,13,421.166667,0.0,0.028205,0.8,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1
204,2,40.0,0,0.0,5,558.500000,0.0,0.028571,0.0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0
4392,0,0.0,0,0.0,16,1599.266667,0.0,0.013333,0.0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


###  Estandarización para regresión logística

**Este es el orden correcto:** primero se divide el dataset; después, se ajusta el escalador únicamente con los datos de entrenamiento. El conjunto de prueba se transforma con el mismo escalador.

Solo estandarizamos las variables numéricas originales: conservamos los indicadores binarios de One-Hot con valores 0 y 1.

Para Random Forest utilizaremos `X_train_cod` y `X_test_cod` sin escalar. Para regresión logística podremos utilizar las versiones con sufijo `_scaled`.

In [29]:
columnas_escalar = [c for c in variables_numericas if c != 'PageValues']

sc = StandardScaler()
X_train_scaled = X_train_cod.copy()
X_test_scaled = X_test_cod.copy()

X_train_scaled[columnas_escalar] = sc.fit_transform(X_train_cod[columnas_escalar])
X_test_scaled[columnas_escalar] = sc.transform(X_test_cod[columnas_escalar])

print('Forma para regresión logística:', X_train_scaled.shape, X_test_scaled.shape)
print('Forma para Random Forest:', X_train_cod.shape, X_test_cod.shape)
display(X_train_scaled[columnas_escalar].head())

Forma para regresión logística: (9764, 72) (2441, 72)
Forma para Random Forest: (9764, 72) (2441, 72)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,SpecialDay
7469,1.689206,0.676238,0.381750,-0.243015,0.220868,-0.080395,-0.448703,-0.800192,-0.311837
11804,-0.699498,-0.458067,-0.397879,-0.243015,-0.046900,-0.158842,-0.448703,-0.739774,-0.311837
3250,-0.699498,-0.458067,-0.397879,-0.243015,-0.426237,-0.408089,-0.448703,-0.287446,3.689185
204,-0.102322,-0.235218,-0.397879,-0.243015,-0.604748,-0.336795,-0.448703,-0.279507,-0.311837
4392,-0.699498,-0.458067,-0.397879,-0.243015,-0.359295,0.203501,-0.448703,-0.609743,-0.311837


##  Preparación para **regresión lineal**

**Objetivo:** predecir `ProductRelated_Duration` en segundos, sin utilizarla como característica de entrada. Tampoco utilizaremos `Revenue`, ya que aquí buscamos estimar tiempo y no usar un resultado futuro de compra. Excluimos inicialmente `PageValues` por precaución.


In [30]:
columnas_excluir_regresion = [
    'Revenue', 'ProductRelated_Duration', 'PageValues',
    'PaginasTotales', 'VisitoInformativas'
]

X_reg = df_trabajo.drop(columns=columnas_excluir_regresion).copy()
y_reg = df_trabajo['ProductRelated_Duration'].copy()

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

X_train_reg_aux = X_train_reg.copy()
X_test_reg_aux = X_test_reg.copy()
for variable in columnas_onehot:
    # Usar las mismas categorías de entrenamiento en ambos conjuntos.
    categorias = sorted(X_train_reg_aux[variable].astype(str).unique())
    X_train_reg_aux[variable] = pd.Categorical(
        X_train_reg_aux[variable].astype(str), categories=categorias
    )
    X_test_reg_aux[variable] = pd.Categorical(
        X_test_reg_aux[variable].astype(str), categories=categorias
    )
X_train_reg_aux['Weekend'] = X_train_reg_aux['Weekend'].astype(int)
X_test_reg_aux['Weekend'] = X_test_reg_aux['Weekend'].astype(int)

X_train_reg_cod = pd.get_dummies(X_train_reg_aux, columns=columnas_onehot, dtype=int, drop_first=True)
X_test_reg_cod = pd.get_dummies(X_test_reg_aux, columns=columnas_onehot, dtype=int, drop_first=True)
X_test_reg_cod = X_test_reg_cod.reindex(columns=X_train_reg_cod.columns, fill_value=0)

print('X_train_reg:', X_train_reg_cod.shape)
print('X_test_reg:', X_test_reg_cod.shape)
print('y_train_reg:', y_train_reg.shape)

X_train_reg: (9764, 64)
X_test_reg: (2441, 64)
y_train_reg: (9764,)


## Preparación para **K-Means**



Como K-Means es no supervisado y aquí se busca describir este conjunto de sesiones, se ajusta el escalador con las sesiones que se agruparán. Esta preparación es distinta de la evaluación sobre un conjunto de prueba de un clasificador.

In [31]:
columnas_kmeans = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates'
]

X_kmeans = df_trabajo[columnas_kmeans].copy()
sc_kmeans = StandardScaler()
X_kmeans_scaled = pd.DataFrame(
    sc_kmeans.fit_transform(X_kmeans),
    columns=X_kmeans.columns,
    index=X_kmeans.index
)

print('Datos listos para K-Means:', X_kmeans_scaled.shape)
display(X_kmeans_scaled.head())

Datos listos para K-Means: (12205, 8)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates
0,-0.702302,-0.460019,-0.398824,-0.246257,-0.696218,-0.628793,3.969402,3.434394
1,-0.702302,-0.460019,-0.398824,-0.246257,-0.673793,-0.595451,-0.450137,1.268054
2,-0.702302,-0.460019,-0.398824,-0.246257,-0.696218,-0.628793,3.969402,3.434394
3,-0.702302,-0.460019,-0.398824,-0.246257,-0.673793,-0.627404,0.654748,2.134590
4,-0.702302,-0.460019,-0.398824,-0.246257,-0.494387,-0.301889,-0.008183,0.184884


##  Preparación básica para **Apriori**





In [32]:
mediana_paginas = df_trabajo['ProductRelated'].median()
mediana_tiempo = df_trabajo['ProductRelated_Duration'].median()
mediana_salida = df_trabajo['ExitRates'].median()

df_apriori = pd.DataFrame({
    'VisitoInformacion': df_trabajo['Informational'] > 0,
    'MuchasPaginasProducto': df_trabajo['ProductRelated'] > mediana_paginas,
    'MuchoTiempoEnProductos': df_trabajo['ProductRelated_Duration'] > mediana_tiempo,
    'SalidaAlta': df_trabajo['ExitRates'] > mediana_salida,
    'FinDeSemana': df_trabajo['Weekend'],
    'VisitanteNuevo': df_trabajo['VisitorType'].eq('New_Visitor'),
    'MesNoviembre': df_trabajo['Month'].eq('Nov'),
    'Compra': df_trabajo['Revenue'],
    'NoCompra': ~df_trabajo['Revenue']
}, index=df_trabajo.index)

print('Formato para Apriori:', df_apriori.shape)
display(df_apriori.head())
print('Frecuencia de cada indicador:')
display((df_apriori.mean() * 100).round(2).sort_values(ascending=False))

Formato para Apriori: (12205, 9)


,VisitoInformacion,MuchasPaginasProducto,MuchoTiempoEnProductos,SalidaAlta,FinDeSemana,VisitanteNuevo,MesNoviembre,Compra,NoCompra
0,False,False,False,True,False,False,False,False,True
1,False,False,False,True,False,False,False,False,True
2,False,False,False,True,False,False,False,False,True
3,False,False,False,True,False,False,False,False,True
4,False,False,True,True,True,False,False,False,True


Frecuencia de cada indicador:


,0
NoCompra,84.37
MuchoTiempoEnProductos,50.00
SalidaAlta,49.65
MuchasPaginasProducto,49.49
MesNoviembre,24.43
FinDeSemana,23.42
VisitoInformacion,21.56
Compra,15.63
VisitanteNuevo,13.87


## Comprobaciones finales

Validamos que todas las matrices de los modelos tengan tipos adecuados, dimensiones coherentes y ausencia de valores faltantes. no entrenados

In [33]:
assert not X_train_cod.isna().any().any()
assert not X_test_cod.isna().any().any()
assert not X_train_scaled.isna().any().any()
assert not X_test_scaled.isna().any().any()
assert X_train_cod.columns.equals(X_test_cod.columns)
assert X_train_reg_cod.columns.equals(X_test_reg_cod.columns)
assert not X_train_reg_cod.isna().any().any()
assert not X_test_reg_cod.isna().any().any()
assert not X_kmeans_scaled.isna().any().any()
assert df_apriori.dtypes.eq(bool).all()

print('Verificación completada.')
print('Clasificación:', X_train_cod.shape, X_test_cod.shape)
print('Regresión:', X_train_reg_cod.shape, X_test_reg_cod.shape)
print('K-Means:', X_kmeans_scaled.shape)
print('Apriori:', df_apriori.shape)

Verificación completada.
Clasificación: (9764, 72) (2441, 72)
Regresión: (9764, 64) (2441, 64)
K-Means: (12205, 8)
Apriori: (12205, 9)


## Guardar el dataset preparado


**Cambiar el lugar donde lo quiran guardar :)**


In [34]:
carpeta_salida = Path('/content/drive/MyDrive/mineriaDatos')
if not carpeta_salida.exists():
    carpeta_salida = Path('.')

nombre_salida = ('online_shoppers_sin_duplicados.csv' if ELIMINAR_DUPLICADOS
                 else 'online_shoppers_con_duplicados.csv')
archivo_salida = carpeta_salida / nombre_salida
df_trabajo.to_csv(archivo_salida, index=False)
print('Dataset de trabajo guardado en:', archivo_salida)
print('Dimensiones:', df_trabajo.shape)

Dataset de trabajo guardado en: /content/drive/MyDrive/mineriaDatos/online_shoppers_sin_duplicados.csv
Dimensiones: (12205, 20)
